# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook guides you through loading, exploring, and analyzing the FAIR^2 dataset using the `mlcroissant` library. All dataset entities are referenced by their `@id` fields, ensuring consistency when handling record sets, fields, and columns.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the FAIR^2 dataset using `mlcroissant`. This step reads the Croissant JSON-LD schema and loads metadata for further inspection.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
# Note: Avoid subscripting or iterating over metadata directly
metadata = dataset.metadata.to_json()
print(f"{metadata['name']}: {metadata['description']}")
print(f"License: {metadata['license']}")
print(f"Published: {metadata['datePublished']}")

## 2. Data Overview
Review the available record sets, fields, and their `@id`s.
Use Croissant schema structure and reference entities by `@id`.

In [ ]:
# Display record sets, fields, and columns
# Reference all entities by their @id per guideline
record_sets = list(dataset.record_sets)
print("Available Record Sets:")
for rs in record_sets:
    print(f"- RecordSet @id: {rs['@id']} | Name: {rs.get('name', '<unknown>')}")
    print("  Fields:")
    for fld in rs['fields']:
        print(f"    - Field @id: {fld['@id']} | Name: {fld.get('name', '<unknown>')}")
        if 'column' in fld:
            print("      Columns:")
            for col in fld['column']:
                print(f"        - Column @id: {col['@id']} | Name: {col.get('name', '<unknown>')}")
    print()
    # Show a sample record for each RecordSet
    print(f"Sample for {rs['@id']}:")
    for rec in dataset.records(record_set=rs['@id']):
        print(rec)
        break
    print('-'*40)

## 3. Data Extraction
Load and convert data from each record set into a DataFrame for analysis. Entities are referenced by their `@id` fields.

In [ ]:
# Extract all record sets by @id dynamically
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}

for rs_id in record_set_ids:
    # Load all records for this record set
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"\nRecordSet @id: {rs_id} | Columns: {df.columns.tolist()}")
    print(df.head(2))

# For further analysis, pick the first record set by @id
main_record_set_id = record_set_ids[0] if record_set_ids else None
print(f"\nUsing main_record_set_id: {main_record_set_id}")
print("Sample columns:", dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps: filter records based on criteria, normalize numeric fields, and group by key attributes.
All fields/columns referenced by their `@id`.

In [ ]:
# Select a numeric field for analysis using its @id
# Inspect columns to select a numeric field (such as 'Age' or 'Interval between cancers')
df = dataframes[main_record_set_id]
print("Available columns:", df.columns.tolist())

# For demonstration, find a likely numeric field @id
numeric_field_id = None
for col in df.columns:
    if 'interval' in col.lower() or 'age' in col.lower():
        numeric_field_id = col
        break
if not numeric_field_id:
    # fallback, pick the first numeric column
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break

print(f"Using numeric field for EDA: {numeric_field_id}")

# Apply filtering: select records where numeric_field > threshold
threshold = 10
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by a categorical field (reference by @id)
# Try finding a groupable column (@id with "location", "msi", "sex", etc.)
group_field_id = None
for col in df.columns:
    if any(x in col.lower() for x in ['location', 'msi', 'sex', 'anatomy', 'status', 'comorbidity']):
        group_field_id = col
        break
if group_field_id:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
    print(f"Grouped data by {group_field_id}:")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields using pandas and matplotlib.
All entities referenced by their `@id`.

In [ ]:
import matplotlib.pyplot as plt

# Distribution of the numeric field
plt.figure(figsize=(7,4))
filtered_df[numeric_field_id].hist(bins=10)
plt.title(f"Distribution of {numeric_field_id} (> {threshold})")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# If group_field_id is found, plot means by group
if group_field_id:
    grouped_df.plot(kind='bar', legend=False)
    plt.title(f"Mean {numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.tight_layout()
    plt.show()

## 6. Conclusion
This notebook demonstrated how to load and explore the FAIR^2 dataset using the `mlcroissant` library, referencing all entities by their `@id` for reproducibility.

- Loaded Croissant metadata and listed available record sets and fields by `@id`
- Extracted and analyzed data using Pandas, filtering and normalizing values for a chosen numeric field
- Grouped and visualized metrics by categorical data for deeper insights

The FAIR^2 dataset supports clinical research into second primary colorectal cancer, with granular fields for demographic, molecular, and pathological analysis.